In [1]:
import os
from typing import Annotated, TypedDict
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

c:\Users\semal\anaconda3\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
# 1. Environment Setup
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("Missing 'OPENAI_API_KEY' in your .env file.")

# Initialize the LLM via OpenRouter
llm = ChatOpenAI(
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="openai/gpt-4o",
    temperature=0
)

In [3]:
# 2. Define the Math Tools
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract the second number from the first number."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide the first number by the second number."""
    if b == 0:
        return "Error: Cannot divide by zero."
    return a / b

tools = [add, subtract, multiply, divide]
llm_with_tools = llm.bind_tools(tools)

In [4]:
# 4. Create the Prompt Template
# This instructs the agent on its exact persona and how to interpret user strings.
calculator_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a strict, precise calculator application. "
     "When the user inputs a mathematical expression as a string (e.g., '1+2/3-5'), "
     "your job is to interpret the expression, break it down, and solve it step-by-step using your provided tools. "
     "Critically important: You MUST follow the standard order of operations (PEMDAS/BODMAS: Multiplication and Division before Addition and Subtraction). "
     "Never do math in your head. Always use tools to compute the intermediate steps. "
     "Once the expression is fully evaluated, return the final calculated value."),
    # MessagesPlaceholder automatically unpacks the 'messages' list from our state, 
    # injecting the human prompt and the entire conversation history.
    MessagesPlaceholder(variable_name="messages"),
])

In [5]:
# 3. Define the State Schema
class AgentState(TypedDict):
    # 'add_messages' ensures that every new turn (User, AI, or Tool) 
    # is appended to the history list automatically.
    messages: Annotated[list[AnyMessage], add_messages]

In [6]:
# 5. Define the Nodes
def call_model(state: AgentState):
    """
    This node takes the history, formats it through the prompt template, 
    and passes it to the LLM to either respond or call a tool.
    """
    # Chain the prompt template directly into the LLM
    chain = calculator_prompt | llm_with_tools
    
    # Invoke the chain with the current message history
    response = chain.invoke({"messages": state["messages"]})
    
    return {"messages": [response]}

# 6. Define Conditional Routing Logic
def should_continue(state: AgentState) -> str:
    messages = state["messages"]
    last_message = messages[-1]
    
    if last_message.tool_calls:
        return "tools"
    return END

tool_node = ToolNode(tools)

In [7]:
# 7. Build and Compile the Graph
workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
workflow.add_edge("tools", "agent")

app = workflow.compile()

In [8]:
if __name__ == "__main__":
    print("Calculator Agent is ready! (Type 'quit' or 'exit' to stop)")
    print("Example input: 1 + 2 * 3 - 4 / 2")
    print("-" * 50)
    
    session_history = []
    
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ["quit", "exit"]:
            break
            
        session_history.append(("user", user_input))
        inputs = {"messages": session_history}
        
        # FIX 3: Track the number of messages printed to ensure we don't miss any,
        # even if a node happens to return multiple messages at once.
        printed_count = len(session_history) - 1 # Subtract 1 so we print the user prompt we just added
        
        for event in app.stream(inputs, stream_mode="values"):
            current_messages = event["messages"]
            
            # Slice the array to only look at messages we haven't printed yet
            new_messages = current_messages[printed_count:]
            for message in new_messages:
                message.pretty_print()
                
            # Update our tracker to the new total length
            printed_count = len(current_messages)
            
        # Update session history with the final resolved state
        session_history = current_messages

Calculator Agent is ready! (Type 'quit' or 'exit' to stop)
Example input: 1 + 2 * 3 - 4 / 2
--------------------------------------------------
================================ Human Message =================================

1 + 2 * 3 - 4 / 2
================================== Ai Message ==================================

To solve the expression `1 + 2 * 3 - 4 / 2`, we need to follow the order of operations (PEMDAS/BODMAS). This means we first handle multiplication and division from left to right, and then addition and subtraction.

1. First, perform the multiplication: `2 * 3`.
2. Then, perform the division: `4 / 2`.
3. Finally, perform the addition and subtraction in order from left to right.

Let's start with the multiplication and division.
Tool Calls:
  multiply (call_l1JbPYsnU2PFXtuy7xfPaLE9)
 Call ID: call_l1JbPYsnU2PFXtuy7xfPaLE9
  Args:
    a: 2
    b: 3
  divide (call_vCkkA9J9lqDfueMdazxFf4TO)
 Call ID: call_vCkkA9J9lqDfueMdazxFf4TO
  Args:
    a: 4
    b: 2
================

KeyboardInterrupt: Interrupted by user